# Notebook 26 — khóa và kiểm toán dữ liệu 5 task × 3 host

Bước chuyển từ T3 sang T4. Giữ a3_dev_n5 làm development, bổ sung ba case có CPU và RAM không suy ra nhau. Mọi case vẫn 25 qubit. Notebook này **không chạy optimizer hay statevector**; nó tái tạo lựa chọn, kiểm toán mô hình và chạy FF/BFD theo Notebook21.

Đọc `N26_README_FIRST.md`. Chạy Run All, lưu notebook đã chạy và gửi ZIP kết quả ở ô cuối. Không chỉnh dữ liệu theo kết quả thuật toán.

In [1]:
from pathlib import Path
import sys
import json
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'src/validation5x3.py').exists()), None)
if root is None:
    raise RuntimeError('Hãy mở notebook trong thư mục repo đã giải nén gói N26.')
if str(root) not in sys.path: sys.path.insert(0, str(root))
from src.validation5x3 import load_dataset, run_audit, DATA_HASH
from IPython.display import display, Markdown
print('Repo:', root)
print('Dataset SHA256:', DATA_HASH)


Repo: c:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement
Dataset SHA256: 19bcd132aa4e240220611111dd81190fa21c84842f3c5f2bfb29de036c290758


## 1. Xem input và protocol đã khóa

Mỗi cặp là `(CPU, RAM)`. Thứ tự t0…t4 và h0…h2 được giữ nguyên cho mọi thuật toán. Tính ngẫu nhiên nằm ở quy tắc chọn mẫu và xáo thứ tự có seed; các con số vẫn là số nguyên nhỏ do giới hạn mô hình.

In [2]:
data = load_dataset(root)
instances = [data['development_instance'], *data['validation_instances']]
for i in instances:
    print(i['instance_id'])
    print('  tasks:', [(t['cpu'],t['ram']) for t in i['tasks']])
    print('  hosts:', [(h['cpu'],h['ram']) for h in i['hosts']])
print('Pool:', data['pool_statistics'])
display(Markdown(data['selection_protocol']))


a3_dev_n5
  tasks: [(1, 1), (1, 1), (1, 1), (1, 1), (1, 2)]
  hosts: [(2, 2), (3, 3), (3, 3)]
v5x3_1
  tasks: [(1, 1), (1, 1), (1, 1), (1, 2), (1, 1)]
  hosts: [(2, 3), (3, 2), (3, 3)]
v5x3_2
  tasks: [(1, 3), (2, 1), (1, 2), (1, 1), (1, 1)]
  hosts: [(2, 2), (3, 3), (2, 3)]
v5x3_3
  tasks: [(3, 1), (1, 1), (1, 2), (1, 1), (1, 1)]
  hosts: [(3, 3), (2, 3), (3, 3)]
Pool: {'raw_multiset_candidates': 25740, 'feasible_multiset_candidates': 752, 'independent_heterogeneous_candidates': 416, 'unique_eligible_after_exclusions': 217, 'excluded_signature_count': 4, 'strata_sizes': [5, 106, 106], 'selection_seed': 20261025}


# Protocol kiểm chứng 5×3 — chốt trước khi chạy QAOA mới

Ngày 25/10/2026. Giữ a3_dev_n5 làm development; bổ sung đúng ba instance kiểm chứng, tất cả 5 task × 3 host và 25 qubit. Đây là mẫu tổng hợp có điều kiện, không đại diện mọi workload thực tế.

## Quy tắc chọn dữ liệu

1. Duyệt hết multiset 5 demand trong {1,2,3}² và multiset 3 capacity trong {2,3}². Không giới hạn tổng demand ≤6 như bộ cũ. Capacity 2/3 giữ đúng hai bit slack mỗi resource/host.
2. Giữ instance có nghiệm, ít nhất hai loại task và hai loại host, đồng thời có assignment qua CPU nhưng trượt RAM và assignment qua RAM nhưng trượt CPU. Hai ràng buộc do đó không suy ra nhau trên toàn bộ 243 assignment.
3. Loại tương đương do hoán vị task/host hoặc đổi CPU↔RAM; loại chữ ký của các input lịch sử 5×3 đã thu được trong snapshot A/B (lưu danh sách loại trừ).
4. Chia ba tầng: optimum=2; optimum=3 với số assignment khả thi ở nửa dưới; optimum=3 ở nửa trên. Chia nửa bằng thứ tự (số khả thi, chữ ký), floor(N/2). Chọn một phần tử mỗi tầng có SHA256({seed:20261025, signature:…}) nhỏ nhất. Không dùng QAOA, FF hay BFD để lọc/chọn. Nghiệm tối ưu chỉ dùng định nghĩa tầng và chấm điểm.
5. Xáo trộn thứ tự task/host một lần bằng Random(20261025+k), k=0,1,2 theo thứ tự tầng; khóa thứ tự cho mọi thuật toán. Không thử nhiều thứ tự rồi lấy kết quả đẹp.

Đây là lựa chọn dự kiến trước kết quả QAOA trên các instance mới, không phải preregistration công khai hay kiểm định mù. Với ba instance và một lần khởi tạo, chỉ kết luận trên từng case; seed huấn luyện không làm tăng số instance độc lập. Optimum=3 khiến mọi placement khả thi đều tối ưu theo mục tiêu số host; phải đọc P_valid/P_opt cùng nhau và giải thích điều này.

## T4 dự kiến được khóa trong bước này

Ba nhánh standard_x, valid_x, valid_dw; penalty domain/capacity=4; exact ideal Aer double CPU, 4 threads, RAM còn tối thiểu4GiB; mean-energy + COBYLA, max_abs Hamiltonian. Mỗi instance/nhánh: p1 bắt đầu [0.3,1.0], tối đa100 lời gọi objective; p2 INTERP từ góc mean-energy tốt nhất p1, tối đa100 lời gọi. rhobeg=1, tol=1e-4; giới hạn thời gian mỗi stage2400giây; báo nếu ngừng sớm. Không chọn góc theo P_opt. Không tái dùng kết quả huấn luyện lịch sử làm một nhánh miễn phí. Tính cả p1, setup, final scoring và mọi simulation ngoài optimizer; cap bằng nhau không đồng nghĩa thời gian bằng nhau.

Trước hết chạy ba nhánh trên development (600 objective evaluations). Sau khi xác minh phần mềm/ngân sách, chạy nguyên cấu hình trên ba instance mới (1800 objective evaluations). Tổng tối đa2400 objective evaluations, cộng preflight/scoring/shots, dự kiến nhiều giờ trên Windows; không hứa thời gian từ một phép đo. Nếu thay phương pháp sau khi xem validation, đánh dấu validation đã dùng phát triển và báo thay đổi. Không tự tăng cap/chọn seed hoặc bỏ case thất bại.

Depth p1 là chuẩn bị có trả chi phí; p2 là kết quả chính với tổng cap200. Báo các bước bị lỗi, nfev/cap, trạng thái dừng, runtime, P_domain/P_valid/P_opt/P_ground/mean-energy. Thêm final sampling1024 shot với seed20261025+k theo thứ tự instance, dùng cùng seed giữa nhánh chỉ để tái lập; exact metrics là chính, counts là minh họa. Chưa thực hiện optimizer trong Notebook26.

## Kiểm toán trước huấn luyện

Notebook26 tái tạo pool và chọn mẫu; kiểm hash/input/thứ tự; vét cạn243 assignment; kiểm QUBO/Ising trên326 vector cơ sở bậc2; giải chính xác slack cho8192 base. Uniform encoded giữ y đo được; random host assignment suy y là control khác. FF/BFD dùng nguyên source Notebook21; MILP vẫn chờ bước T5 với solver/version/status được ghi riêng. Không gọi vét cạn là MILP. Không đo tốc độ thuật toán từ một lần kiểm toán.

Protocol này thay điều khoản “chỉ một exact input cho toàn study” trong kế hoạch cũ. Trong mỗi phép so sánh, mọi phương pháp phải dùng cùng input/hash; kích thước 5×3 cố định từ đầu đến cuối. Artifact T2 và Notebook22–25 vẫn đúng cho a3_dev_n5; không sửa kết quả cũ.


## 2. Tái tạo lựa chọn và kiểm toán

Chọn lại từ toàn pool, so với dữ liệu khóa. Kiểm tra 243 placement bằng hai cách; kiểm đa thức QUBO/Ising trên 326 vector cơ sở; giải chính xác slack cho 8192 base. Các kiểm tra này không cấp đáp án cho FF/BFD hoặc optimizer.

In [3]:
from datetime import datetime, timezone
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
output = root/'data/reference_results/phase5_validation5x3'/f'audit_{stamp}'
result = run_audit(root, output)
assert result['status'] == 'PASS'
assert result['training_evaluations'] == 0
print('PASS | Đã kiểm toán', len(result['cases']), 'case | không huấn luyện')


Audit a3_dev_n5 | 5 tasks x 3 hosts
Audit v5x3_1 | 5 tasks x 3 hosts
Audit v5x3_2 | 5 tasks x 3 hosts
Audit v5x3_3 | 5 tasks x 3 hosts
PASS | Đã kiểm toán 4 case | không huấn luyện


## 3. Đọc bảng đáp án và heuristic

“Không tìm được” của heuristic không có nghĩa bài toán vô nghiệm. Oracle vét cạn cho đáp án chuẩn; MILP chưa chạy trong notebook này. Các số thời gian bên trong JSON chỉ là một lần thực thi, không dùng xếp hạng tốc độ.

In [4]:
rows = ['| Case | Vai trò | Khả thi/243 | Tối ưu/243 | Số host tối ưu | FF | BFD |',
        '|---|---|---:|---:|---:|---|---|']
for c in result['cases']:
    t = c['truth']
    hs = {h['algorithm']: str(h['objective']) if h['found_valid'] else 'Không tìm được'
          for h in c['heuristics']}
    rows.append(f"| {c['instance_id']} | {c['role']} | {t['feasible_assignments']} | {t['optimal_assignments']} | {t['optimum']} | {hs['FirstFit']} | {hs['BFD']} |")
display(Markdown('\n'.join(rows)))
print('Max Ising error:', max(c['max_ising_error'] for c in result['cases']))


| Case | Vai trò | Khả thi/243 | Tối ưu/243 | Số host tối ưu | FF | BFD |
|---|---|---:|---:|---:|---|---|
| a3_dev_n5 | development | 90 | 8 | 2 | 3 | 3 |
| v5x3_1 | prospective_validation | 78 | 4 | 2 | 3 | 3 |
| v5x3_2 | prospective_validation | 1 | 1 | 3 | Không tìm được | Không tìm được |
| v5x3_3 | prospective_validation | 14 | 14 | 3 | 3 | 3 |

Max Ising error: 0.0


## 4. Chứng cứ hai tài nguyên không dư thừa

In một assignment qua CPU nhưng trượt RAM và ngược lại. Danh sách `[h(t0),…,h(t4)]` dùng chỉ số host từ 0. Case development cũ có CPU dư thừa; ba case mới phải có cả hai loại chứng cứ.

In [5]:
for c in result['cases']:
    t = c['truth']
    print(c['instance_id'])
    print('  Qua CPU, trượt RAM:', t['cpu_pass_ram_fail'], '| ví dụ:', t['cpu_pass_ram_fail_witness'])
    print('  Qua RAM, trượt CPU:', t['ram_pass_cpu_fail'], '| ví dụ:', t['ram_pass_cpu_fail_witness'])
    if c['role'] == 'prospective_validation':
        assert t['cpu_pass_ram_fail'] > 0 and t['ram_pass_cpu_fail'] > 0
print('Lưu ý: case optimum=3 có P_opt=P_valid theo mục tiêu min_active_hosts.')


a3_dev_n5
  Qua CPU, trượt RAM: 80 | ví dụ: [0, 0, 1, 1, 1]
  Qua RAM, trượt CPU: 0 | ví dụ: None
v5x3_1
  Qua CPU, trượt RAM: 92 | ví dụ: [0, 0, 1, 1, 1]
  Qua RAM, trượt CPU: 12 | ví dụ: [0, 0, 0, 1, 2]
v5x3_2
  Qua CPU, trượt RAM: 49 | ví dụ: [0, 1, 0, 1, 2]
  Qua RAM, trượt CPU: 7 | ví dụ: [1, 0, 2, 0, 2]
v5x3_3
  Qua CPU, trượt RAM: 6 | ví dụ: [0, 1, 2, 2, 2]
  Qua RAM, trượt CPU: 124 | ví dụ: [0, 0, 1, 0, 1]
Lưu ý: case optimum=3 có P_opt=P_valid theo mục tiêu min_active_hosts.


## 5. Xuất bằng chứng

ZIP gồm dữ liệu, audit, protocol và snapshot source để truy vết. Bước tiếp theo là T4: huấn luyện có đối chứng trên development, sau đó áp dụng nguyên cấu hình cho ba case mới. Tổng cap dự kiến 2400 objective evaluations, chưa gồm scoring/preflight. Không suy hiệu quả QAOA từ bảng oracle/heuristic này.

In [6]:
import hashlib
import zipfile
from src.shot_noise_campaign import save_json
source_dir = output/'source/src'
source_dir.mkdir(parents=True)
for name in result['source_sha256']:
    (source_dir/name).write_bytes((root/'src'/name).read_bytes())
(output/'protocol.md').write_text(data['selection_protocol'], encoding='utf-8')
files = sorted(p for p in output.rglob('*') if p.is_file())
checksums = {p.relative_to(output).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest() for p in files}
save_json(output/'SHA256.json', checksums)
archive = output.with_suffix('.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(output.rglob('*')):
        if p.is_file(): z.write(p, p.relative_to(output).as_posix())
print('GỬI LẠI ZIP:', archive)
print('Đồng thời Save notebook 26 đã chạy rồi gửi lại. Không cần gửi toàn bộ repo.')


GỬI LẠI ZIP: c:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement\data\reference_results\phase5_validation5x3\audit_20261025T045140381848Z.zip
Đồng thời Save notebook 26 đã chạy rồi gửi lại. Không cần gửi toàn bộ repo.
